In [2]:
import tensorflow as tf
import keras

In [3]:
from keras.models import Sequential
from keras.layers import Conv2D , MaxPool2D , Flatten , Dense , Dropout
from tensorflow.keras.utils import to_categorical

In [4]:
import os
import cv2  #computer vision cv2images would be reaed

In [5]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model

In [7]:
image_size = 224

path1 = r"E:\Imarticus\images\Car"
cate = ['Undamaged' , 'Damaged' ]

input_image=[] 
for i in cate:
    folders = os.path.join(path1,i)
    label   = cate.index(i)  # we need to tell software which image is of catand which is ofdog
    for image in os.listdir(folders):
        image_path = os.path.join(folders , image)
        image_array =cv2.imread(image_path) # using the cv2 i am reading the image and storing in variable image
        image_array = cv2.resize(image_array , (image_size, image_size))  # resizing each image to50*50
        input_image.append([image_array, label])
        

In [8]:
np.random.shuffle(input_image)
X = []
Y = []
for X_values , labels in input_image:
    X.append(X_values)
    Y.append(labels)

# seprate pixcells (images) and Y (0,1)`

In [9]:
len(X)

18992

In [10]:
len(X) *.8

15193.6

In [11]:
x_train = X[0:15000]
y_train = Y[0:15000]

x_test = X[15000::]
y_test = Y[15000::]

In [12]:
x_train = np.array(x_train)
y_train = np.array(y_train)
x_test = np.array(x_test)

In [13]:
x_train= x_train/255 # this is to normalize the data 
x_test= x_test/255

In [14]:
vgg_base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

for layer in vgg_base.layers:
    layer.trainable = False

x = vgg_base.output
x = Flatten()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(2, activation="softmax")(x)

model = Model(inputs=vgg_base.input, outputs=output)

In [ ]:
model.compile( optimizer= 'adam' , loss = 'sparse_categorical_crossentropy' , metrics = ['accuracy'])
model.fit(x_train , y_train , epochs = 5 , validation_split=.2 , batch_size= 64)

Epoch 1/5
126/188 ━━━━━━━━━━━━━━━━━━━━ 7:27 7s/step - accuracy: 0.9676 - loss: 0.1063

In [ ]:
pred = model.predict(x_test)

In [ ]:
pred_classes = pred.argmax(axis = 1 ) # argmax  on each row

In [ ]:
from  sklearn.metrics import confusion_matrix , classification_report
print(confusion_matrix(y_test, pred_classes))
print(classification_report(y_test , pred_classes))

In [ ]:
# model.save("car_detaction_RS4.h5")